<a href="https://colab.research.google.com/github/moist234/ECON3916-Statistical-Machine-Learning/blob/main/lab-12/Lab12_OLS_Hedonic_Pricing_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/moist234/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)
df

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent
...,...,...,...,...,...
995,389368.45,2744.4,4.1,10.49,Excellent
996,380314.99,2512.3,8.7,6.18,Average
997,230005.36,1765.8,73.6,6.57,Average
998,360943.75,2021.1,16.6,5.48,Poor


In [2]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula =  'Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit'

In [3]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula,data=df)
results = model.fit()
print(results.summary())


                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     1087.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.73e-313
Time:                        20:16:57   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.415e+04
Df Residuals:                     996   BIC:                         2.417e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept            8.898e+04   6

In [4]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [5]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df["Home_Value"],y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,341.60


In [6]:
import plotly.graph_objects as go
import numpy as np

# --- EXTRACT RESIDUALS FROM STATSMODELS RESULTS OBJECT ---
# results.fittedvalues pulls the y-hat (predicted) values from the OLS fit
fitted = results.fittedvalues

# results.resid extracts the raw residual vector: e = y - y_hat
residuals = results.resid

# --- OUTLIER CLASSIFICATION (2 STD THRESHOLD) ---
# Calculate the standard deviation of the residual distribution
resid_std = residuals.std()

# Boolean mask: True where |residual| exceeds 2 standard deviations
is_outlier = residuals.abs() > 2 * resid_std

# Split into two series for distinct visual treatment
fitted_outliers    = fitted[is_outlier]
residuals_outliers = residuals[is_outlier]
fitted_normal      = fitted[~is_outlier]
residuals_normal   = residuals[~is_outlier]

# --- BUILD FIGURE ---
fig = go.Figure()

# Layer 1: Normal residuals (steel blue)
fig.add_trace(go.Scatter(
    x=fitted_normal,
    y=residuals_normal,
    mode='markers',
    marker=dict(color='steelblue', size=6, opacity=0.65),
    name='Normal Residual'
))

# Layer 2: Outlier residuals (crimson) — plotted on top
fig.add_trace(go.Scatter(
    x=fitted_outliers,
    y=residuals_outliers,
    mode='markers',
    marker=dict(color='crimson', size=9, opacity=0.9, symbol='circle-open-dot', line=dict(width=2)),
    name='Outlier (>2σ)'
))

# Layer 3: Zero reference line — ideal residuals hug this line
fig.add_hline(
    y=0,
    line=dict(color='black', dash='dash', width=1.5),
    annotation_text='Zero Error Baseline',
    annotation_position='top left'
)

# Layer 4: ±2σ boundary bands for visual reference
fig.add_hline(y= 2 * resid_std, line=dict(color='crimson', dash='dot', width=1), opacity=0.4)
fig.add_hline(y=-2 * resid_std, line=dict(color='crimson', dash='dot', width=1), opacity=0.4)

fig.update_layout(
    title=dict(text='Residual Forensics Dashboard — Zillow OLS Model', font=dict(size=18)),
    xaxis_title='Fitted Values (Predicted Home Value $)',
    yaxis_title='Residuals (Actual − Predicted)',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99),
    height=550
)

fig.show()

# --- OUTLIER REPORT ---
print(f"Total observations  : {len(residuals)}")
print(f"Outliers flagged    : {is_outlier.sum()} ({is_outlier.mean()*100:.1f}% of data)")
print(f"Residual Std Dev    : ${resid_std:,.0f}")
print(f"2σ Threshold        : ±${2*resid_std:,.0f}")

Total observations  : 1000
Outliers flagged    : 36 (3.6% of data)
Residual Std Dev    : $42,363
2σ Threshold        : ±$84,726
